# Customer Service Records Analysis Notebook


## Data Cleaning

In [1]:
# Import the necessary libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3

In [2]:
# Loading the dataset
df = pd.read_csv('call_center_data.csv')
df.head()

,id,customer_name,sentiment,csat_score,call_timestamp,reason,city,state,channel,response_time,call duration in minutes,call_center
0,DKK-57076809-w-055481-fU,Analise Gairdner,Neutral,7.0,10/29/20,Billing Question,Detroit,Michigan,Call-Center,Within SLA,17,Los Angeles/CA
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,NaN,10/5/20,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore/MD
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,NaN,10/4/20,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles/CA
3,ZJI-96807559-i-620008-m7,Noreen Lafflina,Very Negative,1.0,10/17/20,Billing Question,Portland,Oregon,Chatbot,Within SLA,12,Los Angeles/CA
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,NaN,10/17/20,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles/CA


### Helper Functions to get a Snapshot of Data

In [3]:
def data_explorer(df: pd.DataFrame):
 
    nrows = df.shape[0]
    ncols = df.shape[1]
    
    colnames = list(df.columns)
    dtypes = df.dtypes.astype(str).tolist()
    
    cols_and_types_df = pd.DataFrame(
        {'Column Name': colnames,
        'Data Type': dtypes}
    )
        
    information = f"""
- The dataset has {nrows} rows and {ncols} columns.

Columns Overview:

{cols_and_types_df}"""

    
    return information
    
#----------------------------------------------------------------------------------------------------------------------------    

def nulls_overview(df: pd.DataFrame):
    colnames = list(df.columns)
    nulls = [df[colname].isna().sum() for colname in colnames]
    
    total_df_nulls = sum(nulls)
    nulls_percentage = (total_df_nulls / df.shape[0]) * 100

    
    cols_and_nulls_df = pd.DataFrame(
        {'Column Name': colnames,
        'Missing Values': nulls}
    )
    
    
  
    nulls_info = f"""
- {nulls_percentage:.0f}% of values across the entire dataframe are missing.

Missing/null values Overview:
    
{cols_and_nulls_df}"""
    
    return nulls_info

#---------------------------------------------------------------------------------------------------------------------------- 
def unique_categories(df: pd.DataFrame):
    categorical_vars = df.select_dtypes(include = 'object')
    
    unique_categories_dict = dict()
    
    for column in categorical_vars.columns:
        unique_categories_dict[column] = set(categorical_vars[column].unique().tolist())
        
    category_info = ""
    
    for colname, value_list in unique_categories_dict.items():
        category_info += f"""
        
- Column : {colname}
- Unique Values : 
                {value_list}
                
                """
    return category_info


In [4]:
#Calling the helper functions

cc_exploration = data_explorer(df)
cc_nulls = nulls_overview(df)
cc_categories = unique_categories(df)

print(cc_exploration)
print()
print(cc_nulls)
print()


- The dataset has 32941 rows and 12 columns.

Columns Overview:

                 Column Name Data Type
0                         id    object
1              customer_name    object
2                  sentiment    object
3                 csat_score   float64
4             call_timestamp    object
5                     reason    object
6                       city    object
7                      state    object
8                    channel    object
9              response_time    object
10  call duration in minutes     int64
11               call_center    object


- 63% of values across the entire dataframe are missing.

Missing/null values Overview:
    
                 Column Name  Missing Values
0                         id               0
1              customer_name               0
2                  sentiment               0
3                 csat_score           20670
4             call_timestamp               0
5                     reason               0
6                

Majority of the columns are of the **categorical** type, whilst few are **numeric.** Casting datatype needs to be done for one of the columns.

In [5]:
#Change column data type from object to datetime
df['call_timestamp'] = pd.to_datetime(df['call_timestamp']).dt.normalize()


C:\Users\user\AppData\Local\Temp\ipykernel_26328\3156644133.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['call_timestamp'] = pd.to_datetime(df['call_timestamp']).dt.normalize()


In [6]:
df.drop_duplicates()
df.shape

(32941, 12)

**No duplicates were found** since the dataframe maintained the same shape as before.

In [7]:
#Make a copy of original df
original_df = df.copy()

original_df.head(5)

,id,customer_name,sentiment,csat_score,call_timestamp,reason,city,state,channel,response_time,call duration in minutes,call_center
0,DKK-57076809-w-055481-fU,Analise Gairdner,Neutral,7.0,2020-10-29,Billing Question,Detroit,Michigan,Call-Center,Within SLA,17,Los Angeles/CA
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,NaN,2020-10-05,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore/MD
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,NaN,2020-10-04,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles/CA
3,ZJI-96807559-i-620008-m7,Noreen Lafflina,Very Negative,1.0,2020-10-17,Billing Question,Portland,Oregon,Chatbot,Within SLA,12,Los Angeles/CA
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,NaN,2020-10-17,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles/CA


### Handling Missing Values

Our data explorer function revealed that only the customer satisfaction score (csat) field had NaN values. There are a number of approaches for imputing missing numeric quantities.

In [8]:
nan_df = df[df['csat_score'].isnull()]
nan_df

,id,customer_name,sentiment,csat_score,call_timestamp,reason,city,state,channel,response_time,call duration in minutes,call_center
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,NaN,2020-10-05,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore/MD
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,NaN,2020-10-04,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles/CA
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,NaN,2020-10-17,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles/CA
7,TWX-27007918-I-608789-Xw,Krysta de Tocqueville,Positive,NaN,2020-10-21,Billing Question,New York City,New York,Chatbot,Below SLA,37,Los Angeles/CA
8,XNG-44599118-P-344473-ZU,Oran Lifsey,Very Negative,NaN,2020-10-03,Billing Question,Dallas,Texas,Email,Below SLA,37,Baltimore/MD
...,...,...,...,...,...,...,...,...,...,...,...,...
32934,LHY-72749170-E-641171-8o,Gabriellia Maguire,Very Negative,NaN,2020-10-15,Billing Question,Pocatello,Idaho,Chatbot,Within SLA,19,Los Angeles/CA
32936,BRM-96715111-h-155613-wO,Othelia Ouldcott,Neutral,NaN,2020-10-30,Billing Question,Oklahoma City,Oklahoma,Web,Within SLA,13,Denver/CO
32937,UJH-96531654-y-074703-H4,Tasha Cubbinelli,Negative,NaN,2020-10-07,Billing Question,Crawfordsville,Indiana,Chatbot,Within SLA,42,Baltimore/MD
32938,WDS-58440679-I-064360-TT,Margaux Slaten,Negative,NaN,2020-10-04,Billing Question,Lehigh Acres,Florida,Chatbot,Within SLA,30,Baltimore/MD


Let's group by sentiment and see the multiple records. The intuition is that customers with a sentiment would have an identical corresponding satisfaction score.


In [9]:
nan_df['sentiment'].value_counts()

sentiment
Negative         6975
Neutral          5490
Very Negative    3750
Positive         2436
Very Positive    2019
Name: count, dtype: int64

For customers who have a missing satisfaction score, the most frequent sentiment was **Negative.**

In [10]:
#group by sentiment and take median of csat score

grouped_sentiment = df.groupby('sentiment', as_index = False)['csat_score'].median()
grouped_sentiment

,sentiment,csat_score
0,Negative,5.0
1,Neutral,6.0
2,Positive,8.0
3,Very Negative,2.0
4,Very Positive,9.0


In [11]:
grouped_sentiment_labels = grouped_sentiment['sentiment'].values.tolist()
grouped_csat_score = grouped_sentiment['csat_score'].values.tolist()

zipped_dict = dict(zip(grouped_sentiment_labels, grouped_csat_score))


for sentiment, median_score in zipped_dict.items():
    df.loc[df['sentiment'] == sentiment, 'csat_score'] = df['csat_score'].fillna(zipped_dict[sentiment])

### Splitting Location into City and State Columns

In [12]:
#Split location column based on delimiter
def location_split(location):
    return pd.Series(location.split("/", 1))

df[['center_city', 'center_state']] = df['call_center'].apply(location_split)
df.drop(columns=["call_center"], inplace=True)

df.head()

,id,customer_name,sentiment,csat_score,call_timestamp,reason,city,state,channel,response_time,call duration in minutes,center_city,center_state
0,DKK-57076809-w-055481-fU,Analise Gairdner,Neutral,7.0,2020-10-29,Billing Question,Detroit,Michigan,Call-Center,Within SLA,17,Los Angeles,CA
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,9.0,2020-10-05,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore,MD
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,5.0,2020-10-04,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles,CA
3,ZJI-96807559-i-620008-m7,Noreen Lafflina,Very Negative,1.0,2020-10-17,Billing Question,Portland,Oregon,Chatbot,Within SLA,12,Los Angeles,CA
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,9.0,2020-10-17,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles,CA


In [13]:
df["center_state"].unique()

array(['CA', 'MD', 'CO', 'IL'], dtype=object)

In [14]:

abbr_dict = {
    'CA': 'California',
    'MD': 'Maryland', 
    'CO': 'Colorado',
    'IL': 'Illinois'
}


df['center_state'] = df['center_state'].map(abbr_dict)

df.head()

,id,customer_name,sentiment,csat_score,call_timestamp,reason,city,state,channel,response_time,call duration in minutes,center_city,center_state
0,DKK-57076809-w-055481-fU,Analise Gairdner,Neutral,7.0,2020-10-29,Billing Question,Detroit,Michigan,Call-Center,Within SLA,17,Los Angeles,California
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,9.0,2020-10-05,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore,Maryland
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,5.0,2020-10-04,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles,California
3,ZJI-96807559-i-620008-m7,Noreen Lafflina,Very Negative,1.0,2020-10-17,Billing Question,Portland,Oregon,Chatbot,Within SLA,12,Los Angeles,California
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,9.0,2020-10-17,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles,California


In [15]:
us_state_abbrev_dict = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
    "California": "CA", "Colorado": "CO", "Connecticut": "CT",
    "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN",
    "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY",
    "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
    "Mississippi": "MS", "Missouri": "MO", "Montana": "MT",
    "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH",
    "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
    "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA",
    "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
    "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
    "Vermont": "VT", "Virginia": "VA", "Washington": "WA",
    "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY"
}

df["customer_state_abbrev"] = df["state"].map(us_state_abbrev_dict)

In [16]:
#final columns cleanup

# df = df.drop(columns = ['id', 'customer_name'])
df = df.rename(columns = {"call duration in minutes": "call_duration_mins", "call_timestamp": "date",
                         "city": "customer_city", "state": "customer_state"})

df.columns.tolist()

['id',
 'customer_name',
 'sentiment',
 'csat_score',
 'date',
 'reason',
 'customer_city',
 'customer_state',
 'channel',
 'response_time',
 'call_duration_mins',
 'center_city',
 'center_state',
 'customer_state_abbrev']

In [17]:
df.head()

,id,customer_name,sentiment,csat_score,date,reason,customer_city,customer_state,channel,response_time,call_duration_mins,center_city,center_state,customer_state_abbrev
0,DKK-57076809-w-055481-fU,Analise Gairdner,Neutral,7.0,2020-10-29,Billing Question,Detroit,Michigan,Call-Center,Within SLA,17,Los Angeles,California,MI
1,QGK-72219678-w-102139-KY,Crichton Kidsley,Very Positive,9.0,2020-10-05,Service Outage,Spartanburg,South Carolina,Chatbot,Within SLA,23,Baltimore,Maryland,SC
2,GYJ-30025932-A-023015-LD,Averill Brundrett,Negative,5.0,2020-10-04,Billing Question,Gainesville,Florida,Call-Center,Above SLA,45,Los Angeles,California,FL
3,ZJI-96807559-i-620008-m7,Noreen Lafflina,Very Negative,1.0,2020-10-17,Billing Question,Portland,Oregon,Chatbot,Within SLA,12,Los Angeles,California,OR
4,DDU-69451719-O-176482-Fm,Toma Van der Beken,Very Positive,9.0,2020-10-17,Payments,Fort Wayne,Indiana,Call-Center,Within SLA,23,Los Angeles,California,IN


### Feature Engineering

In [18]:
#Observing the Date Column
date_exploration = df.copy()
df = date_exploration.sort_values(by='date').reset_index(drop = True)

df.head()

,id,customer_name,sentiment,csat_score,date,reason,customer_city,customer_state,channel,response_time,call_duration_mins,center_city,center_state,customer_state_abbrev
0,KVF-52237801-m-994764-1l,Harlen Aspole,Neutral,8.0,2020-10-01,Billing Question,Buffalo,New York,Email,Within SLA,43,Denver,Colorado,NY
1,LHF-79321231-o-604882-TQ,Alisun Sturmey,Positive,8.0,2020-10-01,Billing Question,Charlottesville,Virginia,Email,Above SLA,13,Chicago,Illinois,VA
2,GAG-90946129-V-874791-hc,Salmon Cacacie,Very Negative,1.0,2020-10-01,Billing Question,Houston,Texas,Chatbot,Within SLA,7,Denver,Colorado,TX
3,CVU-92106898-1-319738-Gj,Amalle Coggen,Negative,5.0,2020-10-01,Billing Question,Mobile,Alabama,Email,Within SLA,32,Chicago,Illinois,AL
4,WWV-49935360-J-983682-qr,Aurelie Lovstrom,Very Negative,2.0,2020-10-01,Billing Question,Washington,District of Columbia,Chatbot,Below SLA,16,Los Angeles,California,NaN


In [19]:
df['month'] = df['date'].dt.month
df['month'].unique()

array([10])

In [20]:
df['day'] = df['date'].dt.day
df['day'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [21]:
df.to_csv('clean_call_center_data.csv', index = False)

## Questions to Ask about the Data

* How were the number of calls distributed on a weekly basis?
* What was the average duration of calls that had a sentiment level equal to or less than neutral? 
* Rank each response status by satisfaction scores, and then by duration.
* For customers living in the same location as the call center, what was the average duration?

In [ ]:
#Make a copy of the dataframe ready for sql analysis
sql_df = df.copy()
sql_df.head()

In [ ]:
conn = sqlite3.connect(":memory:")
df.to_sql("call_centre", conn, index=False, if_exists="replace")

In [ ]:
df_head_query = '''
SELECT * 
FROM call_centre
LIMIT 5
'''
df_head = pd.read_sql_query(df_head_query, conn)
df_head

## Question 1 - How were the number of calls or interactions distributed on a weekly basis?

In [ ]:
query1 = '''

WITH raw_week AS(

SELECT strftime('%W', Date) AS week, COUNT(id) AS numberofCalls
FROM call_centre
GROUP BY week
)

SELECT CASE
       WHEN week = '39' THEN 'Week 1'
       WHEN week = '40' THEN 'Week 2'
       WHEN week = '41' THEN 'Week 3'
       WHEN week = '42' THEN 'Week 4'
       WHEN week = '43' THEN 'Week 5'

END AS weekofMonth,
      numberofCalls

FROM raw_week

'''
weekly_interactions = pd.read_sql_query(query1, conn)

weekly_interactions

In [ ]:
plt.plot(weekly_interactions['weekofMonth'], weekly_interactions['numberofCalls'], color='blue', linewidth=3)
plt.xlabel('Week')
plt.ylabel('Number of Calls')
plt.title('Customer Calls on a Weekly Basis')
plt.grid(True)


# Show the plot
plt.show()

## Question 2 - What was the average duration of calls that had a sentiment level equal to or less than neutral?

In [ ]:
query2 = '''

WITH sentiment_scale AS(
SELECT sentiment, 
    CASE  
    WHEN sentiment = 'Very Positive' THEN 5
    WHEN sentiment = 'Positive' THEN 4
    WHEN sentiment = 'Neutral' THEN 3
    WHEN sentiment = 'Negative' THEN 2
    WHEN sentiment = 'Neutral' THEN 3
    WHEN sentiment = 'Negative' THEN 2
    WHEN sentiment = 'Very Negative' THEN 1
END AS sentimentScore,

    duration
FROM call_centre
)

SELECT sentimentScore, ROUND(AVG(duration),3) AS averageDuration
FROM sentiment_scale
WHERE sentimentScore <= 3
GROUP BY sentimentScore
ORDER BY averageDuration DESC
'''

avg_by_sentiment = pd.read_sql_query(query2, conn)
avg_by_sentiment

## Question 3 - Get top 3 calls for response status of 'Above SLA' ranked by satisfaction scores (descending), and then by duration (ascending).

In [ ]:
query3 = '''

WITH response_status_rank AS(
SELECT *,
       DENSE_RANK() OVER (PARTITION BY responseStatus ORDER BY satisfactionScore DESC, duration ASC) 
       AS call_rank
       
FROM call_centre
WHERE responseStatus = 'Above SLA'
)

SELECT call_rank, customerName, customerCity, responseStatus, satisfactionScore, duration
FROM response_status_rank
WHERE call_rank <=3

'''
response_status_rank_df = pd.read_sql_query(query3, conn)
response_status_rank_df

## Question 4 - For customers living in the same location as the call center, what was their most common reason of contact?

In [ ]:
query4 = '''


SELECT reason, COUNT(*) AS reason_count
FROM call_centre
WHERE customerCity = centerCity
GROUP BY reason
ORDER BY reason_count DESC
LIMIT 1
'''
location = pd.read_sql_query(query4, conn)
location

# Results & Business Recommendations

# Next Steps